# NB05 — Sequence Generation and Sampling
**Project 1.1 · Phase E**

## Purpose and scope
This notebook loads the **frozen best checkpoints** from NB03 (LSTM) and NB04 (causal Transformer), samples sequences under the four prescribed decoding settings, and saves FASTA files plus run metadata. It does **not** train models or use the held-out test set.

The final setting is **10,000 sequences for every sampler and model** (up to 80,000 total). A 100-sequence smoke test is available only to check that paths and checkpoints work; those outputs are not final results.

> **Data provenance:** the default is the existing `SRR3544218` pilot run, which is in the curated VHH run list. The shared Drive `v1` corpus is the known all-six-run merge and is not verified as VHH-only. Before final VHH-specific claims, rebuild the curated VHH-only corpus, retrain both models on that version, and update the version/run IDs below.

## Before running

1. Run NB03 and NB04 first; both checkpoints must use the same dataset version and split.
2. In Colab, mount Drive and set `AI_PROJECT_DIR` to the shared `AI_Project_Shared` root. Locally, the repository paths are used.
3. Set `DATA_VERSION` to the exact folder used in training. By default, run IDs follow NB03/NB04's convention: `lstm-notebook-<DATA_VERSION>` and `transformer-notebook-<DATA_VERSION>`.
4. Confirm the data and run manifests describe a chain-pure VHH corpus before making biological claims. Do not use the test split for generation or sampler selection.

The shared `sampling.generate` implementation is used instead of a notebook-specific sampler. It prevents `<PAD>` and `<BOS>` from being sampled, applies the tested top-k/nucleus filtering, uses recorded per-batch RNG seeds, and returns one EOS-termination flag per sequence.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd


def find_repo_root(start):
    for parent in (Path(start).resolve(), *Path(start).resolve().parents):
        if (parent / "utils.py").is_file() and (parent / "training.py").is_file():
            return parent
    return None

ROOT_DIR = find_repo_root(Path.cwd())
if ROOT_DIR is None and Path("/content").exists():
    clone_target = Path("/content/Recherche_AI")
    if not clone_target.exists():
        subprocess.run(["git", "clone", "-b", "main", "https://github.com/AhmadBelhaj4/Recherche_AI.git", str(clone_target)], check=True)
    ROOT_DIR = find_repo_root(clone_target)
if ROOT_DIR is None:
    raise FileNotFoundError("Open the notebook from the Recherche_AI repository, or clone it first.")

# Optional Colab Drive root; set before importing get_paths().
AI_PROJECT_DIR_OVERRIDE = None  # Example: "/content/drive/MyDrive/AI_Project_Shared"
if AI_PROJECT_DIR_OVERRIDE:
    os.environ["AI_PROJECT_DIR"] = str(AI_PROJECT_DIR_OVERRIDE)
sys.path.insert(0, str(ROOT_DIR))
from utils import get_paths, get_device, set_seed
from training import load_checkpoint
from sampling import generate

PATHS = get_paths()
DEVICE = get_device()
DATA_VERSION = os.environ.get("DATA_VERSION_OVERRIDE") or os.environ.get("DATA_VERSION") or "SRR3544218"
MODEL_RUN_IDS = {
    "LSTM": f"lstm-notebook-{DATA_VERSION}",
    "Transformer": f"transformer-notebook-{DATA_VERSION}",
}
SEED = 42
MAX_GEN_LEN = 140        # maximum decoded amino-acid residues; EOS can stop earlier
BATCH_SIZE = 64          # recorded because the sampler seeds each batch
QUICK_TEST = False       # True -> 100 sequences/cell, for a smoke test only
N_SEQUENCES = 100 if QUICK_TEST else 10_000

DATA_DIR = PATHS["processed"] / DATA_VERSION
for required in (DATA_DIR / "manifest.json", DATA_DIR / "split_manifest.json", DATA_DIR / "train.csv"):
    if not required.exists():
        raise FileNotFoundError(f"Missing {required}; run NB01/NB02 and check DATA_VERSION.")
with open(DATA_DIR / "manifest.json", encoding="utf-8") as f:
    DATA_MANIFEST = json.load(f)
with open(DATA_DIR / "split_manifest.json", encoding="utf-8") as f:
    SPLIT_MANIFEST = json.load(f)
if DATA_MANIFEST.get("data_version") != DATA_VERSION:
    raise ValueError("DATA_VERSION does not match manifest.json")
MAX_GEN_LEN = int(DATA_MANIFEST["parameters"]["max_len"])

if DATA_VERSION == "v1":
    DATA_PROVENANCE_NOTE = "Known all-six-run merge; chain type unverified for VHH-only claims."
    print("WARNING: shared v1 is the known all-six-run merge; do not report it as VHH-only.")
elif DATA_VERSION == "SRR3544218":
    DATA_PROVENANCE_NOTE = "Pilot single run SRR3544218 (curated VHH run); not the final three-run VHH-only rebuild."
    print("Pilot: SRR3544218 is one curated VHH run, not the final three-run VHH-only rebuild.")
else:
    DATA_PROVENANCE_NOTE = "Manually verify chain purity from the data manifest and source records."
    print("Verify chain purity from the data manifest and source records before biological interpretation.")

set_seed(SEED)
print(f"Repository: {ROOT_DIR}\nData: {DATA_DIR}\nModels: {PATHS['models']}\nDevice: {DEVICE}")
print(f"Version={DATA_VERSION} | seed={SEED} | sequences/model/sampler={N_SEQUENCES:,} | max residues={MAX_GEN_LEN}")

## 1. Load the frozen models and define decoding

The checkpoint loader reconstructs the architecture from the saved checkpoint configuration. If a checkpoint is missing, stop and fix the run ID/path; do not silently generate from random weights.

An autoregressive model starts with `<BOS>` and predicts one residue at a time. `<EOS>` stops generation. Reaching the maximum length without emitting `<EOS>` is recorded as **not EOS-terminated**; the sequence remains in the output and is assessed rather than silently discarded.

| Sampler | Definition | Main interpretation |
|---|---|---|
| Greedy | Select the maximum-probability next token | Deterministic collapse baseline. |
| Top-k 10, T=0.8 | Sharpen logits with temperature 0.8; sample among the 10 highest-probability tokens | Restricts the low-probability tail. |
| Nucleus p=0.95, T=1.0 | Sample from the smallest ranked prefix whose cumulative probability reaches 0.95 | Adaptive candidate set retaining most probability mass. |
| Nucleus p=0.80, T=1.2 | Flatten logits with temperature 1.2; retain the smallest prefix reaching 0.80 | More stochastic temperature, smaller nucleus mass. |

Temperature rescales logits before sampling: lower values sharpen, higher values flatten. Diversity is not automatically quality; NB06 reports multiple separate metrics.

In [ ]:
loaded_models = {}
for model_name, run_id in MODEL_RUN_IDS.items():
    expected = PATHS["models"] / run_id / "best.pt"
    if not expected.exists():
        raise FileNotFoundError(f"Checkpoint missing: {expected}. Check the run ID and AI_PROJECT_DIR.")
    model, checkpoint = load_checkpoint(run_id, DEVICE, which="best")
    model.eval()
    loaded_models[model_name] = {
        "run_id": run_id,
        "model": model,
        "checkpoint": checkpoint,
        "checkpoint_path": expected,
    }
    print(f"{model_name:<12} run={run_id} | epoch={checkpoint.get('epoch')} | config={checkpoint.get('config')}")

SAMPLING_CONFIGS = {
    "greedy": {"greedy": True, "temperature": 1.0, "top_k": 0, "top_p": 1.0},
    "topk10_T08": {"greedy": False, "temperature": 0.8, "top_k": 10, "top_p": 1.0},
    "nucleus_p95_T10": {"greedy": False, "temperature": 1.0, "top_k": 0, "top_p": 0.95},
    "nucleus_p80_T12": {"greedy": False, "temperature": 1.2, "top_k": 0, "top_p": 0.80},
}

## 2. Generate and save auditable outputs

For each model × sampler, the next cell writes one FASTA plus a JSON sidecar. FASTA headers contain model, sampler, record number, decoded length, and `eos=0/1`. Sidecars record the exact checkpoint/run ID, data version, seed, batch size, decoding parameters, requested/returned sample count, EOS rate, and length summary. A combined `generation_manifest.json` links all eight outputs.

Expected final output: **8 FASTA files × 10,000 sequences**. If `QUICK_TEST=True`, rerun with it set to `False` before reporting final metrics.

In [ ]:
GEN_DIR = PATHS["results"] / "generated" / DATA_VERSION
GEN_DIR.mkdir(parents=True, exist_ok=True)
created_utc = datetime.now(timezone.utc).isoformat(timespec="seconds")
generation_manifest = {
    "created_utc": created_utc,
    "data_version": DATA_VERSION,
    "dataset_provenance_note": DATA_PROVENANCE_NOTE,
    "dataset_manifest_created_utc": DATA_MANIFEST.get("created_utc"),
    "dataset_git_commit": DATA_MANIFEST.get("git_commit"),
    "split_method": SPLIT_MANIFEST.get("method"),
    "seed": SEED,
    "n_sequences_per_model_per_config": N_SEQUENCES,
    "quick_test_only": QUICK_TEST,
    "max_gen_length_residues": MAX_GEN_LEN,
    "batch_size": BATCH_SIZE,
    "sampling_configs": SAMPLING_CONFIGS,
    "models": {},
}
summary_rows = []

for model_name, model_info in loaded_models.items():
    model_slug = model_name.lower()
    model_dir = GEN_DIR / model_slug
    model_dir.mkdir(parents=True, exist_ok=True)
    model_record = {
        "run_id": model_info["run_id"],
        "checkpoint_path": str(model_info["checkpoint_path"]),
        "checkpoint_config": model_info["checkpoint"].get("config", {}),
        "checkpoint_epoch": model_info["checkpoint"].get("epoch"),
        "checkpoint_val_nats": model_info["checkpoint"].get("val_nats"),
        "slug": model_slug,
        "samples": {},
    }

    for sampler, params in SAMPLING_CONFIGS.items():
        print(f"Generating {model_name} / {sampler}: {N_SEQUENCES:,} sequences")
        sequences, eos_flags = generate(
            model=model_info["model"], n=N_SEQUENCES, device=DEVICE,
            max_len=MAX_GEN_LEN, seed=SEED, batch_size=BATCH_SIZE, **params,
        )
        if len(sequences) != N_SEQUENCES or len(eos_flags) != N_SEQUENCES:
            raise RuntimeError("Sampler output count does not match the requested sample count.")

        fasta_path = model_dir / f"{sampler}.fasta"
        with open(fasta_path, "w", encoding="utf-8") as fasta:
            for idx, (seq, did_eos) in enumerate(zip(sequences, eos_flags), start=1):
                fasta.write(f">{model_slug}|{sampler}|{idx:06d}|len={len(seq)}|eos={int(did_eos)}\n{seq}\n")

        lengths = np.asarray([len(s) for s in sequences], dtype=int)
        sidecar = {
            "data_version": DATA_VERSION, "model": model_name, "run_id": model_info["run_id"],
            "checkpoint_path": str(model_info["checkpoint_path"]), "sampler": sampler,
            "parameters": params, "seed": SEED, "batch_size": BATCH_SIZE,
            "max_gen_length_residues": MAX_GEN_LEN, "n_requested": N_SEQUENCES,
            "n_returned": len(sequences), "n_eos_terminated": int(sum(eos_flags)),
            "eos_termination_rate": float(np.mean(eos_flags)),
            "mean_length": float(np.mean(lengths)), "min_length": int(lengths.min()),
            "max_length": int(lengths.max()),
            "fasta_path_from_results": str(fasta_path.relative_to(PATHS["results"])),
            "created_utc": created_utc,
        }
        sidecar_path = model_dir / f"{sampler}.json"
        sidecar_path.write_text(json.dumps(sidecar, indent=2, ensure_ascii=False), encoding="utf-8")
        model_record["samples"][sampler] = {
            "fasta_path_from_results": sidecar["fasta_path_from_results"],
            "metadata_path_from_results": str(sidecar_path.relative_to(PATHS["results"])),
            "n_requested": N_SEQUENCES, "n_returned": len(sequences),
            "n_eos_terminated": sidecar["n_eos_terminated"],
            "eos_termination_rate": sidecar["eos_termination_rate"],
        }
        summary_rows.append({
            "model": model_name, "run_id": model_info["run_id"], "sampler": sampler,
            "n_returned": len(sequences), "eos_termination_rate": sidecar["eos_termination_rate"],
            "mean_length": sidecar["mean_length"], "min_length": sidecar["min_length"],
            "max_length": sidecar["max_length"], "fasta": sidecar["fasta_path_from_results"],
        })
        print(f"Saved {fasta_path} | EOS={sidecar['eos_termination_rate']:.1%} | mean length={sidecar['mean_length']:.1f}")
    generation_manifest["models"][model_name] = model_record

manifest_path = GEN_DIR / "generation_manifest.json"
manifest_path.write_text(json.dumps(generation_manifest, indent=2, ensure_ascii=False), encoding="utf-8")
summary_df = pd.DataFrame(summary_rows)
print(f"\nGeneration manifest: {manifest_path}")
print(summary_df.to_string(index=False))

## Outputs and interpretation

- FASTA files preserve every returned sample, including short or non-EOS-terminated outputs.
- JSON sidecars and `generation_manifest.json` make model/data/sampling provenance auditable.
- EOS rate and length are generation diagnostics. Canonical alphabet plus a plausible length still does not establish VHH identity, native structure, binding, or experimental developability.

No generation was executed while documenting this notebook. The files are written only when its generation cell is run with the project checkpoints and data available.